# Finland Electricity Price Analysis — Data Cleaning

This notebook prepares the raw data for analysis. It combines yearly exports into 
single continuous files, handles missing values, parses timestamps, and fixes data 
types — producing three clean, analysis-ready CSVs saved to `data/processed/`.

## 1. Generation Mix (ENTSO-E — Actual Generation per Production Type)

**Source:** ENTSO-E Transparency Platform, report 16.1.B&C, bidding zone BZN|FI  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** MW output broken down by production type (nuclear, wind, hydro, fossil, etc.)  
**Why we need it:** Main explanatory variable — testing whether generation mix shifts explain price movements  
**Output file:** `finland_generation_mix_2018_2026.csv`

In [1]:
import pandas as pd
import glob

In [2]:
# find every file matching the pattern "AGGREGATED_*.csv" in data folder
file_list = glob.glob('../data/raw/Actual_Generation_per_Production_Type/AGGREGATED_*.CSV')
print(file_list)

['../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201712312300-201812312300 (1).csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201812312300-201912312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_201912312300-202012312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202012312300-202112312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202112312300-202212312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202212312300-202312312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202312312300-202412312300.csv', '../data/raw/Actual_Generation_per_Production_Type\\AGGREGATED_GENERATION_PER_TYPE_GENERATION_202412312300-202512

In [3]:
# loop through each filename, read it into a table, and collect them in a list
all_years = []
for file in file_list:
    df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited
    all_years.append(df_year)

C:\Users\abhis\AppData\Local\Temp\ipykernel_7696\2201276604.py:4: DtypeWarning: Columns (0: Generation (MW)) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(file, sep=",")   # sep=";" because your ENTSO-E exports are semicolon-delimited


In [4]:
# stack all 8 tables into one combined table
df_price_combined = pd.concat(all_years, ignore_index=True)

In [5]:
# check for duplicate timestamps and product type before trusting the result
duplicates = df_price_combined[df_price_combined.duplicated(subset=["MTU (CET/CEST)", "Production Type"], keep=False)]
print("Number of duplicate rows:", len(duplicates))

Number of duplicate rows: 0


In [6]:
df_price_combined.head()

,MTU (CET/CEST),Area,Production Type,Generation (MW)
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Biomass,873.31
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Biomass,855.71
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Biomass,858.29
3,01/01/2018 03:00:00 - 01/01/2018 04:00:00,BZN|FI,Biomass,874.48
4,01/01/2018 04:00:00 - 01/01/2018 05:00:00,BZN|FI,Biomass,869.99


In [7]:
# Rename columns to clean, consistent snake_case names for easier analysis
df_price_combined.columns = ['timestamp_raw', 'area', 'production_type','generation_mw']

In [8]:
# Null value check
df_price_combined.isnull().sum()

timestamp_raw       0
area                0
production_type     0
generation_mw      40
dtype: int64

In [9]:
# Identify which production types have null values
print(df_price_combined[df_price_combined['generation_mw'].isnull()]['production_type'].value_counts())

production_type
Solar                             12
Fossil Gas                         3
Fossil Hard coal                   3
Fossil Oil                         3
Hydro Run-of-river and pondage     3
Other renewable                    3
Waste                              3
Biomass                            2
Fossil Peat                        2
Nuclear                            2
Other                              2
Wind Onshore                       2
Name: count, dtype: int64


In [10]:
# Sort first so forward fill works in the right direction
df_price_combined = df_price_combined.sort_values(['production_type', 'timestamp_raw'])

In [11]:
# Forward fill nulls within each production type separately
df_price_combined['generation_mw'] =df_price_combined.groupby('production_type')['generation_mw'].ffill()

In [12]:
# Confirm no nulls remain
df_price_combined['generation_mw'].isnull().sum()

np.int64(0)

In [13]:
# Extract start time from the timestamp range string (e.g. "01/01/2018 00:00:00 - 01/01/2018 01:00:00")
# Only the start time is needed — end time is always exactly one hour later and carries no additional information
df_price_combined['timestamp'] = df_price_combined['timestamp_raw'].str.split("-").str[0]

In [14]:
df_price_combined['timestamp']

0          01/01/2018 00:00:00 
1          01/01/2018 01:00:00 
2          01/01/2018 02:00:00 
3          01/01/2018 03:00:00 
4          01/01/2018 04:00:00 
                   ...         
3313795    31/12/2026 22:45:00 
3313796    31/12/2026 23:00:00 
3313797    31/12/2026 23:15:00 
3313798    31/12/2026 23:30:00 
3313799    31/12/2026 23:45:00 
Name: timestamp, Length: 3313800, dtype: object

In [15]:
# Strip any timezone suffix like (CET), (CEST), etc.
df_price_combined['timestamp'] = (
    df_price_combined['timestamp']
    .str.replace(r'\s*\([^)]*\)', '', regex=True)  # removes anything in parentheses
    .str.strip()
)

In [16]:
# Convert to datetime
df_price_combined['timestamp'] = pd.to_datetime(
    df_price_combined['timestamp'],
    format='%d/%m/%Y %H:%M:%S',
    dayfirst=True
)

In [17]:
# Drop timestamp_raw now that clean datetime column has been created
df_price_combined =df_price_combined.drop(columns='timestamp_raw')

In [18]:
# Replace n/e and - with 0, then convert to float
df_price_combined['generation_mw'] = df_price_combined['generation_mw'].replace({'n/e':0, '-':0})

In [19]:
# convert to float
df_price_combined['generation_mw'] = pd.to_numeric(df_price_combined['generation_mw'])

### Final verification before saving

In [20]:
df_price_combined.shape

(3313800, 4)

In [21]:
df_price_combined.dtypes

area                          str
production_type               str
generation_mw             float64
timestamp          datetime64[us]
dtype: object

In [22]:
df_price_combined.isnull().sum()

area               0
production_type    0
generation_mw      0
timestamp          0
dtype: int64

In [23]:
df_price_combined.head()

,area,production_type,generation_mw,timestamp
0,BZN|FI,Biomass,873.31,2018-01-01 00:00:00
1,BZN|FI,Biomass,855.71,2018-01-01 01:00:00
2,BZN|FI,Biomass,858.29,2018-01-01 02:00:00
3,BZN|FI,Biomass,874.48,2018-01-01 03:00:00
4,BZN|FI,Biomass,869.99,2018-01-01 04:00:00


In [24]:
# Step 5: sort chronologically and save

df_price_combined = df_price_combined.sort_values(["timestamp", "production_type"])
df_price_combined.to_csv('../data/processed/finland_generation_mix_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_price_combined))

Saved successfully, total rows: 3313800


## 2. Day-ahead Electricity Price (ENTSO-E — Energy Prices 12.1.D)
**Source:** ENTSO-E Transparency Platform, report 12.1.D, bidding zone BZN|FI  
**Coverage:** 2018–2026, hourly resolution  
**What's in it:** Day-ahead electricity price in EUR/MWh for each hour  
**Why we need it:** Main outcome variable — the price we're trying to explain  

## find every file matching the pattern "AGGREGATED_*.csv" in data folder

In [25]:
# find every file matching the pattern "GUI_*.csv" in data folder

file_list = glob.glob('../data/raw/Energy_Prices/GUI_*.CSV')
print(file_list)

['../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201712312300-201812312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201812312300-201912312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_201912312300-202012312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202012312300-202112312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202112312300-202212312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202212312300-202312312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202312312300-202412312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202412312300-202512312300.csv', '../data/raw/Energy_Prices\\GUI_ENERGY_PRICES_202512312300-202612312300.csv']


In [26]:
# loop through each filename, read it into a table, and collect them in a list
all_years = []
for file in file_list:
    df_year = pd.read_csv(file, sep=",")
    all_years.append(df_year)

In [27]:
# stack all 9 tables into one combined table

df_energy_prices = pd.concat(all_years, ignore_index=True)

In [28]:
df_energy_prices.head()

,MTU (CET/CEST),Area,Sequence,Day-ahead Price (EUR/MWh),Intraday Period (CET/CEST),Intraday Price (EUR/MWh)
0,01/01/2018 00:00:00 - 01/01/2018 01:00:00,BZN|FI,Without Sequence,26.33,NaN,NaN
1,01/01/2018 01:00:00 - 01/01/2018 02:00:00,BZN|FI,Without Sequence,26.43,NaN,NaN
2,01/01/2018 02:00:00 - 01/01/2018 03:00:00,BZN|FI,Without Sequence,26.10,NaN,NaN
3,01/01/2018 03:00:00 - 01/01/2018 04:00:00,BZN|FI,Without Sequence,24.70,NaN,NaN
4,01/01/2018 04:00:00 - 01/01/2018 05:00:00,BZN|FI,Without Sequence,24.74,NaN,NaN


In [29]:
df_energy_prices.shape

(112628, 6)

In [30]:
df_energy_prices.isnull().sum()

MTU (CET/CEST)                     0
Area                               0
Sequence                           0
Day-ahead Price (EUR/MWh)          0
Intraday Period (CET/CEST)    112628
Intraday Price (EUR/MWh)      112628
dtype: int64

In [31]:
df_energy_prices =df_energy_prices.drop(columns=['Intraday Period (CET/CEST)', 'Intraday Price (EUR/MWh)'])

In [32]:
# Rename columns to clean, consistent snake_case names for easier analysis
df_energy_prices.columns = ['timestamp_raw', 'area','sequence' ,'price_eur_mwh']

In [33]:
df_energy_prices.dtypes

timestamp_raw        str
area                 str
sequence             str
price_eur_mwh    float64
dtype: object

In [34]:
# Extract start time from the timestamp range string (e.g. "01/01/2018 00:00:00 - 01/01/2018 01:00:00")
# Only the start time is needed — end time is always exactly one hour later and carries no additional information
df_energy_prices['timestamp'] =df_energy_prices['timestamp_raw'].str.split("-").str[0]

In [35]:
df_energy_prices['timestamp']

0         01/01/2018 00:00:00 
1         01/01/2018 01:00:00 
2         01/01/2018 02:00:00 
3         01/01/2018 03:00:00 
4         01/01/2018 04:00:00 
                  ...         
112623    18/06/2026 22:45:00 
112624    18/06/2026 23:00:00 
112625    18/06/2026 23:15:00 
112626    18/06/2026 23:30:00 
112627    18/06/2026 23:45:00 
Name: timestamp, Length: 112628, dtype: object

In [37]:
# Strip timezone label (CET/CEST) left over after split

df_energy_prices['timestamp'] = df_energy_prices['timestamp'].str.replace(r'\s*\([^)]*\)', '', regex=True).str.strip()

In [40]:
# Convert to datetime

df_energy_prices['timestamp'] = pd.to_datetime(df_energy_prices['timestamp'],
                                               format='%d/%m/%Y %H:%M:%S',
                                               dayfirst=True) 

In [41]:
print(df_energy_prices['timestamp'].dtype)

datetime64[us]


In [43]:
df_energy_prices = df_energy_prices.drop(columns='timestamp_raw')

### Final verification before saving

In [45]:
df_energy_prices.shape

(112628, 4)

In [46]:
df_energy_prices.dtypes

area                        str
sequence                    str
price_eur_mwh           float64
timestamp        datetime64[us]
dtype: object

In [47]:
df_energy_prices.isnull().sum()

area             0
sequence         0
price_eur_mwh    0
timestamp        0
dtype: int64

In [48]:
df_energy_prices.head()

,area,sequence,price_eur_mwh,timestamp
0,BZN|FI,Without Sequence,26.33,2018-01-01 00:00:00
1,BZN|FI,Without Sequence,26.43,2018-01-01 01:00:00
2,BZN|FI,Without Sequence,26.10,2018-01-01 02:00:00
3,BZN|FI,Without Sequence,24.70,2018-01-01 03:00:00
4,BZN|FI,Without Sequence,24.74,2018-01-01 04:00:00


In [50]:
# Step 5: sort chronologically and save

df_energy_prices = df_energy_prices.sort_values('timestamp')
df_energy_prices.to_csv('../data/processed/finland_electricity_price_2018_2026.csv', index=False)
print("Saved successfully, total rows:", len(df_energy_prices))

Saved successfully, total rows: 112628
